# TP 4 - Qualité temporelle dans une architecture décisionnelle
### Spaceship Titanic · Qualité des données · EPSI

**Question du jour :** *les mesures énergétiques reçues sont-elles assez fiables pour produire des indicateurs quotidiens ?*

**Objectifs**
1. Détecter des défauts de **qualité temporelle** : horodatages invalides, doublons, trous et latence.
2. Conserver les observations et les indicateurs de qualité dans **Silver**.
3. Utiliser une **dimension calendrier** pour construire des indicateurs **Gold**, puis visualiser les jours à problème.

```text
CSV énergétique → Bronze (brut) → Silver (mesures + indicateurs DQ)
                                              ↓
                              Gold (qualité quotidienne)
                                      ↑
                             Gold (dimension calendrier)
```

**Continuité avec le TP3 :** on reprend le catalogue `spaceship` et ses schémas `landing`, `bronze`, `silver`, `gold`. On ajoute **une source distincte** : les mesures du vaisseau n'ont pas de `PassengerId`, et on ne les joint pas artificiellement aux tables des passagers.

**Périmètre :** contrôle d'un *historique* 2025, **pas** un flux temps réel, **pas** de Grafana ni de pipelines Lakeflow.

## 0. Préparation

**Prérequis :** avoir exécuté le TP3 et disposer du catalogue `spaceship`.

1. Dans **Workspace**, importez ce notebook dans votre dossier `EPSI/Spaceship`.
2. Dans **Catalog Explorer**, ouvrez le volume `spaceship.landing.sources` créé au TP3.
3. Déposez **un seul fichier**, sous le nom exact : `spaceship_energy_2025_student.csv`.
4. Exécutez la cellule ci-dessous. Elle réutilise les schémas du TP3 et ne touche pas aux tables des passagers.

**Hypothèses explicites :** mesures toutes les **15 minutes** ; du **01/01/2025 00:00** au **31/12/2025 23:45**, soit **96 créneaux par jour** et **35 040 créneaux attendus**. Les horodatages sans fuseau du CSV sont interprétés en **UTC** pour éviter les effets des changements d'heure.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Seul paramètre à modifier si le catalogue « spaceship » n'existe pas.
CATALOGUE = "spaceship"
spark.conf.set("spark.sql.session.timeZone", "UTC")

for schema in ("landing", "bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOGUE}`.`{schema}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOGUE}`.`landing`.`sources`")

VOLUME = f"/Volumes/{CATALOGUE}/landing/sources"
FICHIER = "spaceship_energy_2025_student.csv"
CHEMIN_CSV = f"{VOLUME}/{FICHIER}"

BRONZE = f"{CATALOGUE}.bronze.energy_measurements"
SILVER = f"{CATALOGUE}.silver.energy_measurements"
DIM_DATE = f"{CATALOGUE}.gold.dim_calendar_2025"
GOLD = f"{CATALOGUE}.gold.energy_quality_daily"

if FICHIER not in {f.name for f in dbutils.fs.ls(VOLUME)}:
    raise FileNotFoundError(
        f"Déposez {FICHIER} dans {VOLUME}, puis relancez cette cellule."
    )
print("Source trouvée :", CHEMIN_CSV)
print("Tables du TP4 :", BRONZE, SILVER, DIM_DATE, GOLD, sep="\n - ")

## 1. Bronze : conserver la source

**Exercice 1.** Pourquoi ne suffit-il pas de comparer le nombre de lignes reçues avec les 35 040 mesures théoriques ?

**Démonstration guidée :** charger le CSV **sans inférer les types**, conserver le fichier original dans le volume, puis enregistrer une nouvelle table Delta Bronze. Si la table Bronze existe déjà, elle est **conservée** lors d'une nouvelle exécution, comme dans le TP3.

In [ ]:
if not spark.catalog.tableExists(BRONZE):
    brut = (spark.read
        .option("header", True)
        .option("inferSchema", False)  # valeur brute : tout en texte
        .csv(CHEMIN_CSV))
    assert brut.columns == ["event_time", "received_at", "power_mw"]

    (brut.withColumn("_source_file", F.lit(FICHIER))
         .withColumn("_ingested_at", F.current_timestamp())
         .write.format("delta").mode("ignore")
         .saveAsTable(BRONZE))
    print("Bronze créée.")
else:
    print("Bronze déjà présente : elle est conservée.")

display(spark.table(BRONZE).limit(5))

controle_bronze = spark.sql(f"""
    SELECT COUNT(*) AS lignes,
           SUM(CASE WHEN event_time IS NULL THEN 1 ELSE 0 END) AS event_time_null
    FROM {BRONZE}
""")
display(controle_bronze)


**À vous — Observation de Bronze**

1. Relevez le nombre de lignes reçues et le nombre de lignes dont `event_time` est absent.
2. Le nombre de lignes reçues permet-il, à lui seul, d'affirmer que tous les créneaux de 15 minutes sont présents ? Justifiez votre réponse.
3. Dans **Catalog Explorer**, vérifiez le type des colonnes de la table Bronze. Pourquoi les données n'ont-elles pas encore été converties ?

## 2. Silver : contrôler les mesures

**Exercice 2.** Identifier quatre problèmes distincts :

- **Horodatage invalide** : impossible de situer la mesure dans le temps ;
- **Doublon temporel** : plusieurs mesures avec le même `event_time` ;
- **Retard** : `received_at − event_time` supérieur à **15 minutes** ;
- **Valeur métier invalide** (rappel TP2) : puissance négative.

Le code ci-dessous est **fourni**. Il convertit les types, ajoute des **indicateurs d'audit** et conserve **toutes les lignes**. Attention : `_ingested_at` est la date d'import dans Databricks ; elle ne remplace **pas** le `received_at` de la source.

In [ ]:
fenetre_timestamp = Window.partitionBy("event_time")

mesures_silver = (spark.table(BRONZE)
    .select(
        F.col("event_time").alias("event_time_raw"),
        F.col("received_at").alias("received_at_raw"),
        F.col("power_mw").alias("power_mw_raw"),
        "_source_file", "_ingested_at",
    )
    .withColumn("event_time", F.to_timestamp("event_time_raw", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("received_at", F.to_timestamp("received_at_raw", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("power_mw", F.col("power_mw_raw").cast("double"))
    .withColumn("invalid_event_time", F.col("event_time").isNull())
    .withColumn(
        "latency_min",
        (F.unix_timestamp("received_at") - F.unix_timestamp("event_time")) / F.lit(60.0),
    )
    .withColumn("late_over_15m", F.coalesce(F.col("latency_min") > 15, F.lit(False)))
    .withColumn("_n_same_time", F.count("*").over(fenetre_timestamp))
    .withColumn("duplicate_event_time", F.col("event_time").isNotNull() & (F.col("_n_same_time") > 1))
    .withColumn("negative_power", F.coalesce(F.col("power_mw") < 0, F.lit(False)))
    .drop("_n_same_time")
)

# Rejouable : seule la table Silver de CE TP est remplacée.
(mesures_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(SILVER))

print("Silver enregistrée :", SILVER)

In [ ]:
audit_silver = spark.sql(f"""
    SELECT
        COUNT(*) AS lignes,
        COUNT(DISTINCT event_time) AS horodatages_uniques_valides,
        SUM(CASE WHEN invalid_event_time THEN 1 ELSE 0 END) AS dates_invalides,
        SUM(CASE WHEN duplicate_event_time THEN 1 ELSE 0 END) AS lignes_dans_doublons,
        SUM(CASE WHEN late_over_15m THEN 1 ELSE 0 END) AS retards_sup_15_min,
        SUM(CASE WHEN negative_power THEN 1 ELSE 0 END) AS puissances_negatives
    FROM {SILVER}
""")
display(audit_silver)


# Quelques anomalies à interpréter : aucun enregistrement n'a été supprimé.
display(spark.sql(f"""
    SELECT event_time_raw, received_at, power_mw, latency_min,
           invalid_event_time, duplicate_event_time,
           late_over_15m, negative_power
    FROM {SILVER}
    WHERE invalid_event_time OR duplicate_event_time OR late_over_15m OR negative_power
    ORDER BY event_time, received_at
"""))

**À vous — Interpréter les contrôles Silver**

À partir des deux tableaux affichés :

1. Combien de mesures possèdent un horodatage invalide ? Combien de lignes sont concernées par un horodatage dupliqué ?
2. Combien de mesures ont plus de 15 minutes de retard ? Quelle est leur latence ?
3. Existe-t-il des valeurs de puissance négatives ? S'agit-il d'un problème **temporel** ou d'un problème **métier** ?
4. Une mesure reçue en retard est-elle forcément erronée ?

**À retenir :** on conserve les observations suspectes dans Silver et on les signale avec des indicateurs ; aucune mesure n'est automatiquement supprimée.

## 3. Gold : calendrier et qualité par jour

**Exercice 3.** On attend **96 mesures** à chaque date. Construire une référence temporelle, puis publier une table Gold avec **une ligne par jour** :

| Indicateur Gold | Signification |
|---|---|
| `expected_slots` | Nombre de créneaux attendus (= 96) |
| `present_slots` | Nombre de créneaux ayant **au moins une** mesure |
| `missing_slots` | Créneaux sans mesure **horodatée** |
| `duplicate_excess` | Mesures en trop sur des créneaux dupliqués |
| `late_count` | Mesures reçues avec plus de 15 min de retard |
| `completeness_pct` | `100 × present_slots / expected_slots` |
| `has_temporal_issue` | 1 si au moins un des trois défauts temporels est présent |

Une **dimension calendrier** (`dim_calendar_2025`) contient une ligne par date et une clé `date_key`. La table quotidienne pourra la rejoindre sur cette clé. C'est une **introduction au principe dimensionnel** : le modèle en étoile complet sera étudié au TP5.

**Important :** le *trou temporel* signifie « pas de `event_time` valide pour ce créneau », **pas forcément « aucune mesure physique réalisée »**.

In [ ]:
# 3A. Dimension calendrier : 365 jours en 2025.
spark.sql(f"""
    CREATE OR REPLACE TABLE {DIM_DATE} AS
    SELECT
        CAST(date_format(calendar_date, 'yyyyMMdd') AS INT) AS date_key,
        calendar_date,
        year(calendar_date) AS year,
        month(calendar_date) AS month,
        day(calendar_date) AS day_of_month
    FROM (
        SELECT explode(sequence(
            DATE '2025-01-01', DATE '2025-12-31', INTERVAL 1 DAY
        )) AS calendar_date
    )
""")

display(spark.table(DIM_DATE).orderBy("calendar_date").limit(5))

In [ ]:
# 3B. Grille de référence : 35 040 créneaux attendus (15 min).
# Code fourni : il n'est pas demandé de réécrire ce bloc.
attendus = spark.sql("""
    SELECT explode(sequence(
        TIMESTAMP '2025-01-01 00:00:00',
        TIMESTAMP '2025-12-31 23:45:00',
        INTERVAL 15 MINUTES
    )) AS expected_time
""")

# Une mesure valide est rattachée à son event_time, même si elle arrive en retard.
# Un timestamp dupliqué garde 2 observations pour le même créneau.
par_creneau = (spark.table(SILVER)
    .filter(F.col("event_time").isNotNull())
    .groupBy("event_time")
    .agg(
        F.count("*").alias("n_observations"),
        F.sum(F.col("late_over_15m").cast("int")).alias("n_late"),
    ))

grid = (attendus
    .join(par_creneau, F.col("expected_time") == F.col("event_time"), "left")
    .select(
        F.to_date("expected_time").alias("calendar_date"),
        F.coalesce(F.col("n_observations"), F.lit(0)).alias("n_observations"),
        F.coalesce(F.col("n_late"), F.lit(0)).alias("n_late"),
    )
    .withColumn("missing_slot", (F.col("n_observations") == 0).cast("int"))
    .withColumn("duplicate_extra", F.greatest(F.col("n_observations") - 1, F.lit(0)))
)

# Agrégation au grain JOUR et rattachement à la dimension calendrier.
par_jour = (grid.groupBy("calendar_date")
    .agg(
        F.count("*").alias("expected_slots"),
        F.sum((F.col("n_observations") > 0).cast("int")).alias("present_slots"),
        F.sum("n_observations").alias("received_observations"),
        F.sum("missing_slot").alias("missing_slots"),
        F.sum("duplicate_extra").alias("duplicate_excess"),
        F.sum("n_late").alias("late_count"),
    )
    .withColumn("completeness_pct",
        F.round(F.col("present_slots") * 100.0 / F.col("expected_slots"), 2))
    .withColumn("has_temporal_issue",
        ((F.col("missing_slots") + F.col("duplicate_excess") + F.col("late_count")) > 0).cast("int"))
)

qualite_gold = (par_jour
    .join(spark.table(DIM_DATE).select("date_key", "calendar_date"), "calendar_date", "inner")
    .select("date_key", "expected_slots", "present_slots", "received_observations",
            "missing_slots", "duplicate_excess", "late_count",
            "completeness_pct", "has_temporal_issue"))

(qualite_gold.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD))

print("Dimension :", DIM_DATE, "| Faits quotidiens :", GOLD)

In [ ]:
# 3C. Contrôler la qualité temporelle sur l'année et identifier les journées problématiques
bilan_gold = spark.sql(f"""
    SELECT COUNT(*) AS jours,
           SUM(expected_slots) AS creneaux_attendus,
           SUM(present_slots) AS creneaux_presents,
           SUM(received_observations) AS lignes_horodatees,
           SUM(missing_slots) AS creneaux_absents,
           SUM(duplicate_excess) AS doublons_excedentaires,
           SUM(late_count) AS retards,
           SUM(has_temporal_issue) AS jours_a_probleme
    FROM {GOLD}
""")
display(bilan_gold)


display(spark.sql(f"""
    SELECT d.calendar_date, q.expected_slots, q.present_slots,
           q.missing_slots, q.duplicate_excess, q.late_count, q.completeness_pct
    FROM {GOLD} q
    JOIN {DIM_DATE} d USING (date_key)
    WHERE q.has_temporal_issue = 1
    ORDER BY d.calendar_date
"""))

**À vous — Lire les indicateurs Gold**

1. Combien de jours présentent au moins un défaut temporel ? Relevez les dates concernées.
2. Pour chaque journée signalée, précisez s'il s'agit de créneaux manquants, d'un doublon ou de retards.
3. Une journée peut-elle afficher **100 % de complétude** tout en présentant un autre problème de qualité ? Expliquez.
4. Comparez le nombre de lignes Bronze avec la somme des lignes horodatées de Gold. Comment expliquer un éventuel écart ?

**Petite manipulation :** dans la dernière requête, remplacez `WHERE q.has_temporal_issue = 1` par `WHERE q.late_count > 0`. Quelle journée apparaît ?

## 4. Heatmap : visualiser la qualité du flux

**Exercice 4.** Une case représente un **jour** de 2025 (colonne = jour du mois, ligne = mois) :
- `0` : aucun défaut temporel détecté ;
- `1` : **au moins un** créneau manquant, doublon excédentaire ou retard > 15 min.

Le graphique est **fourni** : repérez les **journées** à investiguer. Une heatmap **binaire** permet de voir les anomalies rares sans laisser plusieurs retards sur une même journée écraser les autres jours sur une échelle continue.

In [ ]:
# Vue destinée à la heatmap : 365 lignes, une par jour.
heatmap_df = spark.sql(f"""
    SELECT d.month AS mois,
           d.day_of_month AS jour,
           q.has_temporal_issue AS jour_a_probleme
    FROM {GOLD} q
    JOIN {DIM_DATE} d USING (date_key)
    ORDER BY mois, jour
""")

# Visualisation automatique dans Databricks
import matplotlib.pyplot as plt

heat = heatmap_df.toPandas()
matrice = (heat.pivot(index="mois", columns="jour", values="jour_a_probleme")
    .reindex(index=range(1, 13), columns=range(1, 32)))

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(matrice, aspect="auto", interpolation="nearest", vmin=0, vmax=1)
ax.set_xticks(range(31), labels=range(1, 32))
ax.set_yticks(range(12), labels=["Jan", "Fév", "Mar", "Avr", "Mai", "Jun",
                                  "Jul", "Aoû", "Sep", "Oct", "Nov", "Déc"])
ax.set_xlabel("Jour du mois")
ax.set_ylabel("Mois")
ax.set_title("Qualité temporelle des mesures — 2025 (0 = OK ; 1 = défaut)")
fig.colorbar(im, ax=ax, ticks=[0, 1], label="Jour avec anomalie temporelle")
fig.tight_layout()
plt.show()

**À vous — Interpréter la heatmap**

1. Quels jours sont signalés ? Retrouvez dans le tableau Gold le type de défaut associé à chacun.
2. La heatmap permet-elle à elle seule de comprendre **pourquoi** une journée est signalée ?
3. Pourquoi certaines cases en fin de mois sont-elles blanches ? S'agit-il d'absences de mesures ?

**Variante Databricks (facultative) :** exécutez `display(heatmap_df)`, puis créez une visualisation **Heatmap** à partir du résultat : `X = jour`, `Y = mois`, `Color = jour_a_probleme`, agrégation `Max`. Le graphique Matplotlib fourni reste la référence du TP.

## Synthèse du TP

1. **Bronze** garde le CSV d'origine et la trace d'ingestion.
2. **Silver** type les données, calcule la **latence** et conserve les mesures **suspectes** avec des indicateurs explicites.
3. **Gold** utilise les **créneaux attendus**, puis la **dimension calendrier**, pour calculer une complétude correcte **sans masquer les doublons**.
4. **Une donnée manquante, un doublon et un retard sont trois défauts différents** ; ils ne se résument pas à un nombre de lignes.
5. **Le monitoring** présenté ici est un **diagnostic sur données historiques**. La fraîcheur « depuis la dernière arrivée » n'est pas un indicateur pertinent pour ce fichier d'archives 2025. La surveillance et les alertes automatiques exigeraient une ingestion récurrente.

**Lien avec le TP5 :** `gold.dim_calendar_2025` illustre la notion de dimension ; `gold.energy_quality_daily` possède un grain explicite (**1 ligne par jour**). Nous approfondirons faits/dimensions et schéma en étoile dans le chapitre suivant.